# 低秩近似与误差

把明确构造的灰度图当矩阵，比较截断重建。图不是外部照片；逐像素的平方误差在显示裁切前计算。

依赖：NumPy、Matplotlib。按顺序运行，先在纸上预测再核对；图内使用英文标签。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


## 先手算两项

diag(4,1) 若只留一项，最小误差平方是多少？


In [ ]:
simple = np.diag([4.,1.])
U,s,Vt = np.linalg.svd(simple,full_matrices=False)
rank_one = (U[:,:1]*s[:1]) @ Vt[:1]
print('秩一近似:',rank_one,'误差平方:',np.linalg.norm(simple-rank_one,'fro')**2)
assert np.allclose(np.linalg.norm(simple-rank_one,'fro')**2,1)


## 构造一张灰度矩阵

包含一块矩形、一条斜带与条纹。先猜哪些形状容易由少数方向表示。


In [ ]:
rows,cols = np.mgrid[0:112,0:128]
image = (.1+.45*((rows>20)&(rows<85)&(cols>25)&(cols<95))
         +.2*(np.abs(rows-cols*.8)<5)
         +.15*np.sin(rows/8)*np.cos(cols/10))
image = np.clip(image,0,1)
U,s,Vt = np.linalg.svd(image,full_matrices=False)
print('原始数字数:',image.size,'前十个奇异值:',s[:10])


## 改变保留项数

预测 k=1、3、10 的损失。保存方向是否比逐格保存更少？


In [ ]:
ranks = [1,3,10]
fig,axes = plt.subplots(1,4,figsize=(12,4),dpi=150)
axes[0].imshow(image,cmap='gray',vmin=0,vmax=1)
axes[0].set_title('Original')
for ax,k in zip(axes[1:],ranks):
    approximate = (U[:,:k]*s[:k]) @ Vt[:k]
    error2 = np.linalg.norm(image-approximate,'fro')**2
    tail2 = np.sum(s[k:]**2)
    assert np.allclose(error2,tail2)
    stored = k*(sum(image.shape)+1)
    print('k:',k,'平方误差:',error2,'存储数字数:',stored)
    ax.imshow(np.clip(approximate,0,1),cmap='gray',vmin=0,vmax=1)
    ax.set_title(f'Rank {k}')
for ax in axes:
    ax.axis('off')
fig.tight_layout()
plt.show()


## 损失曲线

随着 k 增加，剩余误差是否可能变大？


In [ ]:
ks = np.arange(len(s)+1)
tail = np.array([np.sum(s[k:]**2) for k in ks])
assert np.all(np.diff(tail)<=1e-10)
fig,ax = plt.subplots(figsize=(8,4),dpi=140)
ax.plot(ks,tail,color='#2878b5')
ax.set(xlabel='retained rank k',ylabel='squared reconstruction error',title='Loss from omitted directions')
ax.grid(alpha=.2)
plt.show()


## 合上实验，自己解释

指出已知量、未知量、单位与计算目的。用纸上推导解释观察结果，再改变一个参数检查；有限次计算不是一般结论的证明。
